<a href="https://colab.research.google.com/github/coco123-123/ADS401.G.github.io/blob/main/ISA407_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ISA407 — Assignment 2
## Sentiment classification with SmolLM2

Use pretrained language models for IMDB classification in JAX. Model, data and training code are
provided; complete the questions below. This assignment is worth **15%** of the module.

**Released:** 2 November 2026  
**Deadline:** 15 November 2026, 23:59  
**Late penalty:** 5% of available marks per calendar day, up to five days; then zero.

Work individually. Discussion is encouraged, but submit your own code and results. Only submit work
you can explain. Record every AI prompt in the AI usage section at the end of this notebook.

**Submit this notebook**, with all outputs saved, as `p02s_<StudentID>.ipynb` through Learning Mall.

**A video of maximum 10 minutes** is required to explain what you did in Q4.

The 15-minute lab defence is on **26 November 2026** and draws on your submitted notebook.


## Setup
Run the installation cell, then restart the session if pip reports that numpy or any other library
was changed — a half-upgraded runtime is the usual cause of confusing import errors. Prompt and
hyperparameter tuning are optional; select using validation data, never test.


In [ ]:
# GIVEN: environment
import os
os.environ["HF_ENDPOINT"] = "https://huggingface.co"
# If the official endpoint is inaccessible, use "https://hf-mirror.com" above before imports.

# Colab already ships a CUDA-enabled JAX. Do NOT install "jax[cuda]" here: it adds a second CUDA
# plugin, and two plugins claiming device type "cuda" abort with
#   ALREADY_EXISTS: PJRT_Api already exists for device type cuda
# Only if `print(jax.devices())` shows no GPU, run this once in a fresh cell, then restart:
%pip install -q --upgrade jax[cuda]
%pip install -q --upgrade flax optax transformers
# Install other requirements with a compatible numpy version
%pip install -q numpy==2.2.0 datasets scikit-learn scipy pandas matplotlib safetensors

import json, time, gc
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax import traverse_util, serialization
from flax.training.train_state import TrainState
import optax
from transformers import AutoTokenizer
from datasets import load_dataset, Dataset
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file
from sklearn.metrics import classification_report, accuracy_score, f1_score

In [ ]:
# GIVEN: settings
SEED = 42
MAX_LEN = 256
BASE_ID = "HuggingFaceTB/SmolLM2-135M"
INSTRUCT_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
STUDENT_ID = "StudentID"  # replace with your student ID
CHECKPOINT_DIR = "isa407_smollm2_checkpoints"


## Data
Official IMDB split, with a stratified 10% validation set carved from training (seed 42). Reviews are
tokenised by each checkpoint's own BPE tokenizer — no stemming, stop-word removal or hand-written
string processing, because the pretrained vocabulary already covers the text.


In [ ]:
# GIVEN: official split
split = load_dataset("stanfordnlp/imdb")
tv = split["train"].train_test_split(test_size=0.1, seed=SEED, stratify_by_column="label")
raw = {"train": tv["train"], "valid": tv["test"], "test": split["test"]}
TEXTS = {key: list(ds["text"]) for key, ds in raw.items()}
LABELS = {key: np.asarray(ds["label"], np.int32) for key, ds in raw.items()}
pd.DataFrame({"split": list(raw), "reviews": [len(ds) for ds in raw.values()]})


In [ ]:
# GIVEN: tokenise with each checkpoint's own tokenizer
NUM_PROC = min(8, os.cpu_count() or 1)

TOKENIZERS = {"base": AutoTokenizer.from_pretrained(BASE_ID),
              "instruct": AutoTokenizer.from_pretrained(INSTRUCT_ID)}
for tokenizer in TOKENIZERS.values():
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"


def tokenize_batch(batch, tokenizer, max_len):
    # A blank review would tokenise to nothing, leaving no real position to read.
    texts = [text if text.strip() else tokenizer.eos_token for text in batch["text"]]
    encoded = tokenizer(texts, truncation=True, max_length=max_len,
                        padding="max_length", return_tensors="np")
    return {key: np.asarray(encoded[key], dtype=np.int32)
            for key in ("input_ids", "attention_mask")}


def tokenize_split(tokenizer, texts, max_len=MAX_LEN):
    encoded = Dataset.from_dict({"text": texts}).map(
        tokenize_batch, fn_kwargs={"tokenizer": tokenizer, "max_len": max_len},
        batched=True, batch_size=1000, num_proc=NUM_PROC,
        remove_columns=["text"], desc="Tokenising")
    return encoded.with_format("numpy", dtype=np.int32)[:]

DATA = {name: {split: tokenize_split(tk, texts) for split, texts in TEXTS.items()}
        for name, tk in TOKENIZERS.items()}


## Provided model and training code
The encoder is loaded once and **frozen**; only a small head is trained, which is what keeps this
within a modest GPU. Q1 trains a two-label head on the last real token. Q3 trains a vocabulary head
and reads the Yes/No logits at the last real prompt token — a different decision rule from a pooled
classifier, because the answer is a *next token*, not a class score.


In [ ]:
# GIVEN: SmolLM2 encoder and heads
@dataclass(frozen=True)
class LlamaCfg:
    vocab_size: int = 49152
    hidden_size: int = 576
    num_hidden_layers: int = 30
    num_attention_heads: int = 9
    num_key_value_heads: int = 3
    intermediate_size: int = 1536
    rms_norm_eps: float = 1e-05
    rope_theta: float = 100000.0

    @property
    def head_dim(self) -> int:
        return self.hidden_size // self.num_attention_heads

    @property
    def n_rep(self) -> int:
        return self.num_attention_heads // self.num_key_value_heads

class RMSNorm(nn.Module):
    eps: float = 1e-05

    @nn.compact
    def __call__(self, x):
        w = self.param('weight', nn.initializers.ones, (x.shape[-1],))
        var = jnp.mean(jnp.square(x.astype(jnp.float32)), axis=-1, keepdims=True)
        return (x.astype(jnp.float32) * jax.lax.rsqrt(var + self.eps)).astype(x.dtype) * w

def rope_tables(seq_len, head_dim, theta, dtype=jnp.float32):
    inv_freq = 1.0 / theta ** (jnp.arange(0, head_dim, 2, dtype=jnp.float32) / head_dim)
    freqs = jnp.outer(jnp.arange(seq_len, dtype=jnp.float32), inv_freq)
    emb = jnp.concatenate([freqs, freqs], axis=-1)
    return (jnp.cos(emb).astype(dtype), jnp.sin(emb).astype(dtype))

def rotate_half(x):
    x1, x2 = jnp.split(x, 2, axis=-1)
    return jnp.concatenate([-x2, x1], axis=-1)

def apply_rope(x, cos, sin):
    return x * cos[None, :, None, :] + rotate_half(x) * sin[None, :, None, :]

class LlamaAttention(nn.Module):
    cfg: LlamaCfg

    @nn.compact
    def __call__(self, x, cos, sin, mask):
        B, T, _ = x.shape
        hd, H, KV = (self.cfg.head_dim, self.cfg.num_attention_heads, self.cfg.num_key_value_heads)
        q = nn.Dense(H * hd, use_bias=False, name='q_proj')(x).reshape(B, T, H, hd)
        k = nn.Dense(KV * hd, use_bias=False, name='k_proj')(x).reshape(B, T, KV, hd)
        v = nn.Dense(KV * hd, use_bias=False, name='v_proj')(x).reshape(B, T, KV, hd)
        q, k = (apply_rope(q, cos, sin), apply_rope(k, cos, sin))
        k = jnp.repeat(k, self.cfg.n_rep, axis=2)
        v = jnp.repeat(v, self.cfg.n_rep, axis=2)
        q, k, v = (jnp.transpose(a, (0, 2, 1, 3)) for a in (q, k, v))
        att = jnp.einsum('bhqd,bhkd->bhqk', q, k) * hd ** (-0.5)
        att = jnp.where(mask, att, jnp.finfo(att.dtype).min)
        att = jax.nn.softmax(att.astype(jnp.float32), axis=-1).astype(x.dtype)
        out = jnp.einsum('bhqk,bhkd->bhqd', att, v)
        out = jnp.transpose(out, (0, 2, 1, 3)).reshape(B, T, H * hd)
        return nn.Dense(self.cfg.hidden_size, use_bias=False, name='o_proj')(out)

class LlamaMLP(nn.Module):
    cfg: LlamaCfg

    @nn.compact
    def __call__(self, x):
        gate = nn.Dense(self.cfg.intermediate_size, use_bias=False, name='gate_proj')(x)
        up = nn.Dense(self.cfg.intermediate_size, use_bias=False, name='up_proj')(x)
        return nn.Dense(self.cfg.hidden_size, use_bias=False, name='down_proj')(nn.silu(gate) * up)

class LlamaBlock(nn.Module):
    cfg: LlamaCfg

    @nn.compact
    def __call__(self, x, cos, sin, mask):
        h = x + LlamaAttention(self.cfg, name='self_attn')(RMSNorm(self.cfg.rms_norm_eps, name='input_layernorm')(x), cos, sin, mask)
        return h + LlamaMLP(self.cfg, name='mlp')(RMSNorm(self.cfg.rms_norm_eps, name='post_attention_layernorm')(h))

class LlamaEncoder(nn.Module):
    cfg: LlamaCfg

    @nn.compact
    def __call__(self, input_ids, attention_mask):
        B, T = input_ids.shape
        x = nn.Embed(self.cfg.vocab_size, self.cfg.hidden_size, name='embed_tokens')(input_ids)
        cos, sin = rope_tables(T, self.cfg.head_dim, self.cfg.rope_theta, x.dtype)
        causal = jnp.tril(jnp.ones((T, T), dtype=bool))
        keep = attention_mask.astype(bool)[:, None, :]
        mask = causal[None, None, :, :] & keep[:, None, :, :]
        for i in range(self.cfg.num_hidden_layers):
            x = LlamaBlock(self.cfg, name=f'layer_{i}')(x, cos, sin, mask)
        return RMSNorm(self.cfg.rms_norm_eps, name='norm')(x)

def last_token_pool(hidden, attention_mask):
    idx = attention_mask.sum(1).astype(jnp.int32) - 1
    return hidden[jnp.arange(hidden.shape[0]), idx]

class Classifier(nn.Module):
    num_labels: int = 2
    dropout: float = 0.1

    @nn.compact
    def __call__(self, h, train: bool):
        h = nn.Dropout(self.dropout, deterministic=not train)(h)
        return nn.Dense(self.num_labels, name='score')(h)

def llama_params_from_state(sd, cfg):
    T = lambda a: jnp.asarray(np.transpose(a))
    C = lambda a: jnp.asarray(a)
    params = {'embed_tokens': {'embedding': C(sd['model.embed_tokens.weight'])}, 'norm': {'weight': C(sd['model.norm.weight'])}}
    for i in range(cfg.num_hidden_layers):
        b = f'model.layers.{i}.'
        params[f'layer_{i}'] = {'input_layernorm': {'weight': C(sd[b + 'input_layernorm.weight'])}, 'post_attention_layernorm': {'weight': C(sd[b + 'post_attention_layernorm.weight'])}, 'self_attn': {'q_proj': {'kernel': T(sd[b + 'self_attn.q_proj.weight'])}, 'k_proj': {'kernel': T(sd[b + 'self_attn.k_proj.weight'])}, 'v_proj': {'kernel': T(sd[b + 'self_attn.v_proj.weight'])}, 'o_proj': {'kernel': T(sd[b + 'self_attn.o_proj.weight'])}}, 'mlp': {'gate_proj': {'kernel': T(sd[b + 'mlp.gate_proj.weight'])}, 'up_proj': {'kernel': T(sd[b + 'mlp.up_proj.weight'])}, 'down_proj': {'kernel': T(sd[b + 'mlp.down_proj.weight'])}}}
    return params

class SmolLM2Classifier(nn.Module):
    cfg: LlamaCfg
    num_labels: int = 2
    dropout: float = 0.1

    @nn.compact
    def __call__(self, input_ids, attention_mask, train: bool):
        hidden = LlamaEncoder(self.cfg, name="encoder")(input_ids, attention_mask)
        pooled = last_token_pool(hidden, attention_mask)
        return Classifier(self.num_labels, self.dropout, name="classifier")(pooled, train)


class SmolLM2AnswerHead(nn.Module):
    cfg: LlamaCfg

    @nn.compact
    def __call__(self, input_ids, attention_mask, train: bool = False):
        hidden = LlamaEncoder(self.cfg, name="encoder")(input_ids, attention_mask)
        last = last_token_pool(hidden, attention_mask)
        return nn.Dense(self.cfg.vocab_size, use_bias=False, name="answer")(last)


In [ ]:
# GIVEN: training utilities
@dataclass(frozen=True)
class TrainingConfig:
    epochs: int = 1
    batch_size: int = 4
    accum: int = 16
    lr_head: float = 1e-3
    weight_decay: float = 0.01
    warmup_frac: float = 0.1
    grad_clip: float = 1.0

CONFIG = TrainingConfig()
def cross_entropy(logits, labels, smoothing=0.0):
    logp = jax.nn.log_softmax(logits, axis=-1)
    n = logits.shape[-1]
    onehot = jax.nn.one_hot(labels, n)
    target = (1.0 - smoothing) * onehot + smoothing / n
    return -(target * logp).sum(-1).mean()

def make_schedule(total_steps, peak_lr, warmup_frac):
    return optax.warmup_cosine_decay_schedule(init_value=0.0, peak_value=peak_lr, warmup_steps=max(1, int(total_steps * warmup_frac)), decay_steps=total_steps, end_value=0.0)

def batches(arrays, labels, batch_size, accum, rng):
    n = len(labels)
    idx = np.asarray(jax.random.permutation(rng, n))
    span = batch_size * accum
    for s in range(n // span):
        sel = idx[s * span:(s + 1) * span].reshape(accum, batch_size)
        yield {'input_ids': jnp.asarray(arrays['input_ids'][sel]), 'attention_mask': jnp.asarray(arrays['attention_mask'][sel]), 'labels': jnp.asarray(labels[sel])}

def save_params(params, path):
    raw = serialization.msgpack_serialize(jax.tree_util.tree_map(np.asarray, params))
    with open(path, 'wb') as f:
        f.write(raw)
    print(f'saved {len(raw) / 1000000.0:.1f} MB -> {path}')

def load_params(path):
    with open(path, 'rb') as f:
        tree = serialization.msgpack_restore(f.read())
    return jax.tree_util.tree_map(jnp.asarray, tree)

def split_params(params, trainable):
    """Separate the frozen encoder from the head that will be trained."""
    frozen = {key: value for key, value in params.items() if key not in trainable}
    return frozen, {key: params[key] for key in trainable}


def make_train_state(model, trainable, total_steps, lr, config):
    schedule = make_schedule(total_steps, lr, config.warmup_frac)
    tx = optax.chain(optax.clip_by_global_norm(config.grad_clip),
                     optax.adamw(schedule, weight_decay=config.weight_decay))
    return TrainState.create(apply_fn=model.apply, params=trainable, tx=tx)


def head_logits(apply_fn, head, frozen, input_ids, attention_mask, train, rngs=None):
    return apply_fn({"params": {**frozen, **head}}, input_ids, attention_mask, train, rngs=rngs)


def make_train_step(model, frozen, config, loss_fn):
    """One optimiser step over all `accum` micro-batches, vectorised with `jax.vmap`.

    The micro-batch loop is mapped, not unrolled: one batched graph instead of `accum` copies
    of a 30-layer model. Gradients are taken w.r.t. the head only.
    """
    @jax.jit
    def step(state, micro):
        keys = jax.random.split(jax.random.fold_in(jax.random.PRNGKey(0), state.step),
                                micro["labels"].shape[0])

        def one(head, ids, mask, labels, key):
            logits = head_logits(model.apply, head, frozen, ids, mask, True,
                                 rngs={"dropout": key})
            return loss_fn(logits, labels)

        def objective(head):
            losses = jax.vmap(one, in_axes=(None, 0, 0, 0, 0))(
                head, micro["input_ids"], micro["attention_mask"], micro["labels"], keys)
            return losses.mean()

        loss, grads = jax.value_and_grad(objective)(state.params)
        return state.apply_gradients(grads=grads), loss
    return step


def make_infer(model):
    """One jitted forward pass. The model already maps (B, T) to (B, ...), so no vmap needed."""
    @jax.jit
    def infer(head, frozen, input_ids, attention_mask):
        return model.apply({"params": {**frozen, **head}}, input_ids, attention_mask, False)
    return infer


def evaluate_split(infer, head, frozen, data, labels, loss_fn, answer_ids, chunk=128):
    total, correct = 0.0, 0
    for start in range(0, len(labels), chunk):
        rows = slice(start, start + chunk)
        logits = infer(head, frozen, jnp.asarray(data["input_ids"][rows]),
                       jnp.asarray(data["attention_mask"][rows]))
        batch_labels = jnp.asarray(labels[rows])
        total += float(loss_fn(logits, batch_labels)) * len(batch_labels)
        correct += int((score_logits(logits, answer_ids).argmax(-1) == batch_labels).sum())
    return total / len(labels), correct / len(labels)


def fit(step, infer, state, frozen, data, labels, valid_data, valid_labels, loss_fn,
        answer_ids, config, seed=SEED, verbose=True):
    rng = jax.random.PRNGKey(seed)
    steps_per_epoch = max(1, len(labels) // (config.batch_size * config.accum))
    history = {"step": [], "train_loss": [], "valid_loss": [], "valid_acc": []}
    best = {"valid_loss": float("inf"), "params": state.params, "step": -1}
    gstep = 0
    for epoch in range(config.epochs):
        rng, rng_epoch = jax.random.split(rng)
        running, seen = 0.0, 0
        for i, micro in enumerate(batches(data, labels, config.batch_size, config.accum, rng_epoch)):
            state, loss = step(state, micro)
            running += float(loss)
            seen += 1
            gstep += 1
            if (i + 1) % max(1, steps_per_epoch // 4) == 0:
                vloss, vacc = evaluate_split(infer, state.params, frozen, valid_data,
                                             valid_labels, loss_fn, answer_ids)
                history["step"].append(gstep)
                history["train_loss"].append(running / seen)
                history["valid_loss"].append(vloss)
                history["valid_acc"].append(vacc)
                if vloss < best["valid_loss"]:
                    best = {"valid_loss": vloss, "params": state.params, "step": gstep}
                if verbose:
                    print(f"  ep{epoch} step {gstep:>5} | train {running / seen:.4f} "
                          f"| valid {vloss:.4f} | acc {vacc:.4f}")
                running, seen = 0.0, 0
    return state.replace(params=best["params"]), history, best


In [ ]:
# GIVEN: prompting, scoring and reporting

def yes_no_logprobs(logits, answer_ids):
    """(batch, vocab) logits -> (batch, 2) log-probabilities over [No, Yes].
    The two answer logits are normalised against each other, not against the whole vocabulary.
    """
    pair = logits[:, jnp.asarray(answer_ids)]
    return jax.nn.log_softmax(pair, axis=-1)


def answer_loss(logits, labels, answer_ids):
    """Mean -log P(correct answer | prompt) over the batch."""
    return -yes_no_logprobs(logits, answer_ids)[jnp.arange(len(labels)), labels].mean()


def score_logits(logits, answer_ids=None):
    """Decision scores: class logits for Q1, two-way log-probabilities for Q3."""
    return logits if answer_ids is None else yes_no_logprobs(logits, answer_ids)


def prompt_data(prompt_fn, tokenizer, texts, max_len=MAX_LEN):
    """Apply the chat template and keep the assistant-generation suffix.

    Long reviews are shortened from the left; the instructions and suffix are never truncated.
    """
    rows = []
    for text in texts:
        review_ids = tokenizer.encode(text, add_special_tokens=False)[-max_len:]
        while True:
            review = tokenizer.decode(review_ids, skip_special_tokens=True)
            messages = [{"role": "user", "content": prompt_fn(review)}]
            ids = tokenizer.apply_chat_template(messages, tokenize=True,
                                                add_generation_prompt=True, return_dict=False)
            if len(ids) <= max_len:
                rows.append(ids)
                break
            if not review_ids:
                raise ValueError("The prompt alone exceeds max_len; shorten it.")
            review_ids = review_ids[min(len(review_ids), max(1, len(ids) - max_len)):]
    padded = tokenizer.pad({"input_ids": rows}, padding="max_length", max_length=max_len,
                           return_attention_mask=True, return_tensors="np")
    return {key: np.asarray(padded[key], np.int32) for key in ("input_ids", "attention_mask")}


def load_model(language_model=False):
    """Frozen encoder plus a freshly initialised head."""
    cfg = LlamaCfg()
    model = SmolLM2AnswerHead(cfg) if language_model else SmolLM2Classifier(cfg)
    ids = jnp.ones((1, 8), jnp.int32)
    params = model.init(jax.random.PRNGKey(SEED), ids, ids, False)["params"]
    repo = INSTRUCT_ID if language_model else BASE_ID
    weights = load_file(hf_hub_download(repo, "model.safetensors"))
    params["encoder"] = llama_params_from_state(weights, cfg)
    if language_model:
        # The checkpoint ties its output head to the embedding matrix. Copy that transpose into
        # the trainable head, so step 0 reproduces the pretrained LM logits exactly.
        params["answer"] = {
            "kernel": jnp.asarray(params["encoder"]["embed_tokens"]["embedding"]).T}
    return model, params


def batch_logits(infer, head, frozen, data, chunk=128):
    """Score a whole split in jitted chunks; the model batches internally."""
    out = []
    for start in range(0, len(data["input_ids"]), chunk):
        out.append(np.asarray(infer(head, frozen,
                                    jnp.asarray(data["input_ids"][start:start + chunk]),
                                    jnp.asarray(data["attention_mask"][start:start + chunk]))))
    return np.concatenate(out)


def report(name, logits, labels, answer_ids=None):
    scores = np.asarray(score_logits(jnp.asarray(logits), answer_ids))
    pred = scores.argmax(1)
    options = dict(labels=[0, 1], target_names=["negative", "positive"], zero_division=0)
    print("Model", name)
    print(classification_report(labels, pred, digits=4, **options))
    return dict(system=name, accuracy=accuracy_score(labels, pred),
                macro_f1=f1_score(labels, pred, average="macro", zero_division=0),
                report=classification_report(labels, pred, output_dict=True, **options),
                scores=scores, predictions=pred, source="Assignment 2", comparable=True)


def evaluate_untrained(name, data, labels, answer_ids):
    """Score the pretrained checkpoint as it is, before any head training."""
    model, params = load_model(language_model=True)
    frozen, head = split_params(params, ["answer"])
    logits = batch_logits(make_infer(model), head, frozen, data)
    del model, params, frozen, head
    gc.collect()
    jax.clear_caches()
    return report(name, logits, labels, answer_ids)


def train_head(name, train_data, train_labels, valid_data, valid_labels, answer_ids=None,
               config=CONFIG, directory=CHECKPOINT_DIR):
    language_model = answer_ids is not None
    model, params = load_model(language_model)
    frozen, head = split_params(params, ["answer"] if language_model else ["classifier"])
    loss_fn = (lambda logits, y: answer_loss(logits, y, answer_ids)) if language_model else cross_entropy
    steps = len(train_labels) // (config.batch_size * config.accum) * config.epochs
    state = make_train_state(model, head, steps, config.lr_head, config)
    started = time.time()
    state, history, best = fit(make_train_step(model, frozen, config, loss_fn), make_infer(model),
                               state, frozen, train_data, train_labels, valid_data, valid_labels,
                               loss_fn, answer_ids, config)
    jax.block_until_ready(state.params)
    os.makedirs(directory, exist_ok=True)
    checkpoint = os.path.join(directory, name + ".msgpack")
    save_params(state.params, checkpoint)          # head only: a few MB, not the encoder
    run = dict(name=name, checkpoint=checkpoint, language_model=language_model,
               answer_ids=list(answer_ids) if answer_ids else None, history=history,
               valid_loss=best["valid_loss"], minutes=(time.time() - started) / 60)
    with open(os.path.join(directory, name + ".json"), "w", encoding="utf-8") as f:
        json.dump(run, f, indent=2)
    del model, params, frozen, state, best
    gc.collect()
    jax.clear_caches()
    return run


def evaluate_run(run, data, labels):
    model, params = load_model(run["language_model"])
    params = {**params, **load_params(run["checkpoint"])}
    frozen, head = split_params(params, ["answer"] if run["language_model"] else ["classifier"])
    logits = batch_logits(make_infer(model), head, frozen, data)
    del model, params, frozen, head
    gc.collect()
    jax.clear_caches()
    return report(run["name"], logits, labels, run["answer_ids"])


def show_history(run):
    h = run["history"]
    plt.plot(h["step"], h["train_loss"], label="train")
    plt.plot(h["step"], h["valid_loss"], label="validation")
    plt.title(run["name"])
    plt.xlabel("Optimiser step")
    plt.ylabel("Cross-entropy")
    plt.legend()
    plt.show()


## Reporting
Report test-set precision, recall, F1 and support for both classes, accuracy, and macro/weighted
averages to four decimals, as in Labs 2–4. Select with validation, then report the test set once.


## Q1. Linear probe on the frozen encoder (20 marks)

Train only the two-label head on the last real token; leave the encoder frozen. Show the loss
curves and the test classification report. Say in your analysis what freezing costs and what it saves.


1.   10 marks for correct implementation (i.e.., calling the right function)

2.   10 marks for showing reasonable run result







In [ ]:
# TODO 1
def train_classifier_head(train_data, train_labels, valid_data, valid_labels):
    raise NotImplementedError("TODO 1")

run_A = train_classifier_head(DATA["base"]["train"], LABELS["train"],
                              DATA["base"]["valid"], LABELS["valid"])
show_history(run_A)
result_A = evaluate_run(run_A, DATA["base"]["test"], LABELS["test"])


## Q2. Design the prompt and evaluate it (30 marks)

Write a prompt that asks whether a review is positive and demands exactly `Yes` or `No`. Implement
`answer_tokens`, returning `[id("No"), id("Yes")]`, and handle the failure modes: a tokenizer without
a chat template, and an answer that encodes to more than one token. Raise a clear error rather than
returning something silently wrong.

**Hint: you need to understnad how the NO_YES_IDS is used**

Then build the prompted **validation** inputs and evaluate the pretrained Instruct model as it
stands, without training. Report the prompt, the two token ids and the validation accuracy.


1.   10 marks for sentiment_prompt implementation

2.   10 marks for answers_toke implementation

3.   10 marks for showing reasonable result


In [ ]:
# TODO 2
def sentiment_prompt(review):
    raise NotImplementedError("TODO 2: write your prompt")


def answer_tokens(tokenizer):
    """Return [No id, Yes id]; each answer must be exactly one token."""
    raise NotImplementedError("TODO 2: validate and return the two answer token ids")

print(sentiment_prompt(TEXTS["train"][0]))
NO_YES_IDS = answer_tokens(TOKENIZERS["instruct"])
print(dict(zip(("No", "Yes"), NO_YES_IDS)))

PROMPT_VALID = prompt_data(sentiment_prompt, TOKENIZERS["instruct"], TEXTS["valid"])
result_zero_shot = evaluate_untrained("C0-prompt-zero-shot", PROMPT_VALID, LABELS["valid"], NO_YES_IDS)


## Q3. Fine-tune the Instruct head on Yes/No (20 marks)

Now process the **training** reviews through the same prompt and train only the vocabulary head of
`SmolLM2-135M-Instruct`, with `-log P(answer | prompt)` for the correct answer. Use the validation
prompts from Q2 to select the checkpoint, then report the test set and compare with Q1 and Q2.


1.   10 marks for correct implementation (i.e.., calling the right function)

2.   10 marks for showing reasonable run results



In [ ]:
# TODO 3
def fine_tune_instruct(train_texts, train_labels, valid_texts, valid_labels, answer_ids):
    raise NotImplementedError("TODO 3: prompt the training data, then train the head only")

run_C = fine_tune_instruct(TEXTS["train"], LABELS["train"], TEXTS["valid"], LABELS["valid"], NO_YES_IDS)
show_history(run_C)
PROMPT_TEST = prompt_data(sentiment_prompt, TOKENIZERS["instruct"], TEXTS["test"])
result_C = evaluate_run(run_C, PROMPT_TEST, LABELS["test"])


In [ ]:
# GIVEN: compare with previous labs on the same official test set.
# The Lab 2 rows are reference values from the course notebook; replace them with your own.
# Leave a score as None if you did not run that lab, and set comparable=False if the split differs.
# Q2 is scored on validation, not test: its prompt is chosen before the model is trained, so
# reading test there would leak the set Q3 is reported on. The label says which split it is.
LAB_RESULTS = [
    dict(system="Lab 2 BoW", accuracy=None, macro_f1=None, source="Lab 2", comparable=True),
    dict(system="Lab 2 TF-IDF", accuracy=None, macro_f1=None, source="Lab 2", comparable=True),
    dict(system="Lab 3 word2vec", accuracy=None, macro_f1=None, source="Lab 3", comparable=True),
    dict(system="Lab 4 LSTM", accuracy=None, macro_f1=None, source="Lab 4", comparable=True),
]
NEW_RESULTS = [("Q1 linear probe (test)", result_A),
               ("Q2 zero-shot prompt (valid)", result_zero_shot),
               ("Q3 Instruct head (test)", result_C)]
columns = ["system", "accuracy", "macro_f1", "source", "comparable"]
comparison = pd.DataFrame(
    [{key: dict(result, system=name).get(key) for key in columns} for name, result in NEW_RESULTS]
    + [{key: row.get(key) for key in columns} for row in LAB_RESULTS], columns=columns)
comparison = comparison.dropna(subset=["accuracy"]).reset_index(drop=True)
comparison[["accuracy", "macro_f1"]] = comparison[["accuracy", "macro_f1"]].astype(float)
display(comparison.sort_values("accuracy", ascending=False).reset_index(drop=True).round(4))


## Q4. Analysis, debugging or algorithmic improvement (30 marks)

This is an open-ended question, so there is no exact marking decomposition.

Students can either

1.   Provide a meaningful empirical analysis

    what are the results, why?
    any further experiments to validate your intuition?

2.   Identify "bugs" in the given code or testing

    what are the bugs, why, what happens if you fix them

3.   Propose an algorithmic improvement

    what motivates your new design , what is your new design
    and how is it better?
    (hyper-parameter tuning does not count)

Students can only pick one option.

**option 2: the bug I know only counts as 15 marks, you can talk to me if you find a real bug. I will let you know whether you can get 30 before you submit.
I will then, publisize the bug, and no other students can get 30 for the same bug.**

You add your answer below, you replace the X with your option id.
You can add as many code boxes or text boxes as you want.

## Q4 Option X




# AI usage
Record your prompts here, verbatim, in order. Responses are not required.


e.g.,
1.   Please finish this assignment for me.
